In [ ]:


import Metal
import time
import ctypes
import re
from datasets import load_dataset
import numpy as np 
from collections import Counter
import gc
import time
import numpy as np
from objc import autorelease_pool


# Use 'strict_small' as identified in your error message
dataset = load_dataset(
    "cambridge-climb/BabyLM", 
    name="strict_small", 
    trust_remote_code=True
)


print(len(dataset['train']))
print(len(dataset['test']))

335858
342095


In [2]:
block_size = 64
embed_dim = 64
hidden_size = 512
batch_size = 128
text = " ".join(sample.lower() + " <eos>" for sample in dataset['train']['text'])#eos is end of sentence token , join all the text samples into one big string "word1 word2 <eos> word3 <eos> ..."
all_words = text.split()

#tokenize
words = text.split()#split breaks the string into list ["word1", "word2", ...]
print(len(words))#9,783,451 words
#vocab


word_counts = Counter(words)#count how many times words appear {"word": number of times it appears}
unique_words = sorted([w for w in set(words) if word_counts[w] >= 50])#keep only words that appear at least 50 times, this reduces vocab size and replaces rare words with <unk> set is used to get unique words, sorted keeps words in line 
unique_words.append("<unk>") # any rare word that appears less than 50 times is replaced with <unk>
vocab_size = len(unique_words)  # 12130
print(vocab_size)

stoi = {w: i for i, w in enumerate(unique_words)}#string to integer mapping {"word": 0, "another_word": 1, ...} used to encode words as integers for training
itos = {i: w for i, w in enumerate(unique_words)}#int to string mapping {0: "word", 1: "another_word", ...} used tto decode back from integers to predicted words


data = np.array([stoi.get(w, stoi["<unk>"]) for w in words])#if word in stoi return the number into the np array if not 0 or unk this is for words which is 9 million 



X, y = [], []
#training task 
for i in range(len(data) - block_size):
    context = data[i:i + block_size]         #[i,i+1, ...,i+7] 
    targets = data[i + 1:i + block_size + 1] #[i+1,i+2, ...,i+8]
    X.append(context)
    y.append(targets)

X = np.array(X)
y = np.array(y)

stride = block_size  # 8
X = X[::stride]#stride is used to not overlap [0,1,2,3,4,5,6,7] [8,9,10,11,12,13,14,15], instead of [0,1,2,3,4,5,6,7] [1,2,3,4,5,6,7,8] 
y = y[::stride]



# Recompute splits after subsampling
split_idx = int(len(X) * 0.9)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

num_samples = X_train.shape[0]#1,1 million rows for each traininng                                                                                         
steps_per_epoch = num_samples // batch_size

print(f"After subsampling: {len(X_train):,} train, {len(X_test):,} test")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Batch size: {batch_size}")

for threshold in [50, 100, 200, 500]:
    kept = [w for w in set(words) if word_counts[w] >= threshold]
    total_unk = sum(1 for w in words if word_counts[w] < threshold)
    pct_unk = total_unk / len(words) * 100
    print(f"threshold={threshold:>4d} → vocab={len(kept):>6,} | {pct_unk:.1f}% of tokens become <unk>")



9783451
12131
After subsampling: 137,579 train, 15,287 test
Steps per epoch: 1074
Batch size: 128
threshold=  50 → vocab=12,130 | 12.9% of tokens become <unk>
threshold= 100 → vocab= 6,798 | 16.7% of tokens become <unk>
threshold= 200 → vocab= 3,752 | 21.0% of tokens become <unk>
threshold= 500 → vocab= 1,584 | 27.8% of tokens become <unk>


In [ ]:
#MINE
def positional_encoding(block_size, embed_dim):
    pos_enc = np.zeros((block_size, embed_dim))
    for pos in range(block_size):
        for i in range(0,embed_dim,2):
            pos_enc[pos,i] = np.sin(pos / (10000 ** (i / embed_dim)))
            pos_enc[pos,i +1] = np.cos(pos / (10000 **(i/embed_dim)))
    return pos_enc

def softmax(x):
    x_shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def layer_norm(x, gamma, beta, eps=1e-5):
    """Forward pass for layer normalization"""
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    out = gamma * x_norm + beta
    cache = (x_norm, var, gamma)  # Save for backward
    return out, cache


def layer_norm_backward(dout, cache):
    x_norm, var, gamma = cache
    axes = tuple(range(dout.ndim - 1))

    dgamma = np.sum(dout * x_norm, axis=axes)
    dbeta = np.sum(dout, axis=axes)

    dx_norm = dout * gamma
    inv_std = 1.0 / np.sqrt(var + 1e-5)

    dx = inv_std * (
        dx_norm
        - np.mean(dx_norm, axis=-1, keepdims=True)
        - x_norm * np.mean(
            dx_norm * x_norm, axis=-1, keepdims=True
        )
    )

    return dx, dgamma, dbeta

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(np.float32)

def cross_entropy_loss(probs, y):
    if probs.ndim != 2 or y.shape != (probs.shape[0],):
        raise ValueError(
            f"Expected probs (batch, vocab) and y (batch,), "
            f"got {probs.shape} and {y.shape}"
        )

    correct_probs = probs[np.arange(y.shape[0]), y]
    return -np.mean(np.log(np.clip(correct_probs, 1e-10, 1.0)))

def clip_gradient(grad, max_norm=1.0):
    norm = np.sqrt(np.sum(grad ** 2))
    if norm > max_norm:
        return grad * max_norm / norm
    return grad


n_blocks =4
epochs =5
block_size = X_train.shape[1]
embed_dim = 64
hidden_size = 512
epochs = 5
batch_size = 32
num_heads = 8
head_dim = embed_dim // num_heads
input_size = block_size * embed_dim
num_samples = X_train.shape[0]
steps_per_epoch = num_samples // batch_size
lr = 0.0003
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
t = 0
#decay = 1e-4


#layer and vocab weight init
W_embed = np.random.randn(vocab_size, embed_dim) * 0.01
#W_1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
#W_2 = np.random.randn(hidden_size, hidden_size) * np.sqrt(2 / hidden_size)
#W_3 = np.random.randn(hidden_size, vocab_size) * np.sqrt(2 / hidden_size)
#b1 = np.zeros(hidden_size)
#b2 = np.zeros(hidden_size)
#b3 = np.zeros(vocab_size)


#layer norm 1
ln1_gamma = np.ones(embed_dim)
ln1_beta = np.zeros(embed_dim)

#Weights
W_ff1 = np.random.randn(embed_dim, hidden_size) * np.sqrt(2 / embed_dim)
b_ff1 = np.zeros(hidden_size)
W_ff2 = np.random.randn(hidden_size, embed_dim) * np.sqrt(2 / hidden_size)
b_ff2 = np.zeros(embed_dim)

#attention layer weight init
W_q = np.random.randn(embed_dim, embed_dim) * 0.02
W_k = np.random.randn(embed_dim, embed_dim) * 0.02
W_v = np.random.randn(embed_dim, embed_dim) * 0.02
W_o = np.random.randn(embed_dim, embed_dim) * 0.02

#layer norm 2

ln2_gamma = np.ones(embed_dim)
ln2_beta = np.zeros(embed_dim)

#out projection after tranformer
W_out = np.random.randn(embed_dim, vocab_size) * np.sqrt(2 / embed_dim)
b_out = np.zeros(vocab_size)

#optimizer momentum and velocity weihgt init
m_W_embed, v_W_embed = np.zeros_like(W_embed), np.zeros_like(W_embed)
m_Wq, v_Wq = np.zeros_like(W_q), np.zeros_like(W_q)
m_Wk, v_Wk = np.zeros_like(W_k), np.zeros_like(W_k)
m_Wv, v_Wv = np.zeros_like(W_v), np.zeros_like(W_v)
m_Wo, v_Wo = np.zeros_like(W_o), np.zeros_like(W_o)
m_ln1_gamma, v_ln1_gamma = np.zeros_like(ln1_gamma), np.zeros_like(ln1_gamma)
m_ln1_beta, v_ln1_beta = np.zeros_like(ln1_beta), np.zeros_like(ln1_beta)
m_W_ff1, v_W_ff1 = np.zeros_like(W_ff1), np.zeros_like(W_ff1)
m_b_ff1, v_b_ff1 = np.zeros_like(b_ff1), np.zeros_like(b_ff1)
m_W_ff2, v_W_ff2 = np.zeros_like(W_ff2), np.zeros_like(W_ff2)
m_b_ff2, v_b_ff2 = np.zeros_like(b_ff2), np.zeros_like(b_ff2)
m_ln2_gamma, v_ln2_gamma = np.zeros_like(ln2_gamma), np.zeros_like(ln2_gamma)
m_ln2_beta, v_ln2_beta = np.zeros_like(ln2_beta), np.zeros_like(ln2_beta)
m_W_out, v_W_out = np.zeros_like(W_out), np.zeros_like(W_out)
m_b_out, v_b_out = np.zeros_like(b_out), np.zeros_like(b_out)

#prevents model from looking at future words when predicting token
CAUSAL_MASK = np.triu(np.ones((block_size, block_size)), k=1) * -1e5

PE = positional_encoding(block_size,embed_dim)#(8,64)
for epoch in range(epochs) :
    indices = np.random.permutation(num_samples)
    X_train_sh = X_train[indices]
    y_train_sh = y_train[indices]

    for batchidx in range(steps_per_epoch):
        start = batchidx * batch_size
        end = start + batch_size
        X_batch = X_train_sh[start:end]#(32,8) 32 rows 8 columns
        y_batch = y_train_sh[start:end]#(32,)

        if y_batch.ndim == 2 and y_batch.shape[1] in (1, block_size):
            y_batch = y_batch[:, -1]

        print(y_batch.shape)
        #Forward
        embedded = W_embed[X_batch] + PE

        #print(embedded.shape)
        #Attention
        Q = np.dot(embedded, W_q).reshape(batch_size, block_size, num_heads, head_dim).transpose(0, 2, 1, 3) #without reshape (32,8,64)
        #print(Q[0][0][0][0])
        #|print('q',Q[0][0][0][0])
        K = np.dot(embedded, W_k).reshape(batch_size, block_size, num_heads, head_dim).transpose(0, 2, 1, 3)
        V = np.dot(embedded, W_v).reshape(batch_size, block_size, num_heads, head_dim).transpose(0, 2, 1, 3)

        scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)
        scores = scores + CAUSAL_MASK
        attW = softmax(scores)
        print(attW.shape)

        attOut = np.matmul(attW, V)
        attOut_reshaped = attOut.transpose(0, 2, 1, 3).reshape(batch_size, block_size, embed_dim)
        projOut = np.dot(attOut_reshaped, W_o)

        #layer norm 1 plus residual
        res1 = embedded + projOut
        ln1_out, ln1_cache = layer_norm(res1, ln1_gamma, ln1_beta)

        #FNN
        ffn_hidden = np.dot(ln1_out, W_ff1) + b_ff1
        ffn_relu = relu(ffn_hidden)
        ffn_out = np.dot(ffn_relu, W_ff2) + b_ff2

        #layer norm 2 plus residual
        res2 = ln1_out + ffn_out
        ln2_out, ln2_cache = layer_norm(res2, ln2_gamma, ln2_beta)

        #output last token
        last_hidden = ln2_out[:, -1, :]
        logits = np.dot(last_hidden, W_out) + b_out
        prob_softmax = softmax(logits)

        #Loss
        loss = cross_entropy_loss(prob_softmax, y_batch)



        #Back
        dlogits = prob_softmax.copy()
        dlogits[np.arange(batch_size), y_batch] -= 1

        dlogits=dlogits / batch_size
        #print("bsdfsadf",dlogits.shape)

        dW_out = np.dot(last_hidden.T, dlogits)
        db_out = np.sum(dlogits, axis=0)
        dlast_hidden = np.dot(dlogits, W_out.T)

        #full sequence
        dln2_out = np.zeros((batch_size, block_size, embed_dim))
        dln2_out[:, -1, :] = dlast_hidden

        #layer norm 2 res back
        dres2, dln2_gamma, dln2_beta = layer_norm_backward(dln2_out, ln2_cache)

        dffn_out = dres2
        dln1_out_res = dres2

        #layer norm  back
        dW_ff2 = np.dot(ffn_relu.reshape(-1, hidden_size).T, dffn_out.reshape(-1, embed_dim))
        db_ff2 = np.sum(dffn_out, axis=(0, 1))
        dffn_relu = np.dot(dffn_out, W_ff2.T)

        dffn_hidden = dffn_relu * relu_derivative(ffn_hidden)

        dW_ff1 = np.dot(ln1_out.reshape(-1, embed_dim).T, dffn_hidden.reshape(-1, hidden_size))
        db_ff1 = np.sum(dffn_hidden, axis=(0, 1))
        dln1_out_ffn = np.dot(dffn_hidden, W_ff1.T)

        # Combine gradients to ln1_out
        dln1_out = dln1_out_res + dln1_out_ffn

        # LayerNorm 1 backward
        dres1, dln1_gamma, dln1_beta = layer_norm_backward(dln1_out, ln1_cache)

        # Residual 1 backward
        dprojOut = dres1
        d_embedded_res = dres1

        # Attention backward
        dW_o = np.dot(attOut_reshaped.reshape(-1, embed_dim).T, dprojOut.reshape(-1, embed_dim))
        d_attOut_combined = np.dot(dprojOut, W_o.T)
        d_attOut = d_attOut_combined.reshape(batch_size, block_size, num_heads, head_dim)
        d_attOut = d_attOut.transpose(0, 2, 1, 3)

        dattW = np.matmul(d_attOut, V.transpose(0, 1, 3, 2))
        dV_heads = np.matmul(attW.transpose(0, 1, 3, 2), d_attOut)

        d_scores = attW * (dattW - np.sum(dattW * attW, axis=-1, keepdims=True))
        d_scores = d_scores / np.sqrt(head_dim)

        dQ_heads = np.matmul(d_scores, K)
        dK_heads = np.matmul(d_scores.transpose(0, 1, 3, 2), Q)

        dQ_res = dQ_heads.transpose(0, 2, 1, 3)
        dQ_res = dQ_res.reshape(batch_size, block_size, embed_dim)

        dK_res = dK_heads.transpose(0, 2, 1, 3)
        dK_res = dK_res.reshape(batch_size, block_size, embed_dim)

        dV_res = dV_heads.transpose(0, 2, 1, 3)
        dV_res = dV_res.reshape(batch_size, block_size, embed_dim)

        dW_q = np.dot(embedded.reshape(-1, embed_dim).T, dQ_res.reshape(-1, embed_dim))
        dW_k = np.dot(embedded.reshape(-1, embed_dim).T, dK_res.reshape(-1, embed_dim))
        dW_v = np.dot(embedded.reshape(-1, embed_dim).T, dV_res.reshape(-1, embed_dim))

        d_embedded = np.dot(dQ_res, W_q.T) + np.dot(dK_res, W_k.T) + np.dot(dV_res, W_v.T)
        d_embedded = d_embedded + d_embedded_res

        dW_embed = np.zeros_like(W_embed)
        np.add.at(dW_embed, X_batch, d_embedded)

        t += 1
        #current_lr = lr / (1 + decay * t)
        current_lr = lr
        for p, grad, m, v in [
            (W_embed, dW_embed, m_W_embed, v_W_embed),
            (W_q, dW_q, m_Wq, v_Wq),
            (W_k, dW_k, m_Wk, v_Wk),
            (W_v, dW_v, m_Wv, v_Wv),
            (W_o, dW_o, m_Wo, v_Wo),
            (ln1_gamma, dln1_gamma, m_ln1_gamma, v_ln1_gamma),
            (ln1_beta, dln1_beta, m_ln1_beta, v_ln1_beta),
            (W_ff1, dW_ff1, m_W_ff1, v_W_ff1),
            (b_ff1, db_ff1, m_b_ff1, v_b_ff1),
            (W_ff2, dW_ff2, m_W_ff2, v_W_ff2),
            (b_ff2, db_ff2, m_b_ff2, v_b_ff2),
            (ln2_gamma, dln2_gamma, m_ln2_gamma, v_ln2_gamma),
            (ln2_beta, dln2_beta, m_ln2_beta, v_ln2_beta),
            (W_out, dW_out, m_W_out, v_W_out),
            (b_out, db_out, m_b_out, v_b_out),
        ]:
            grad = clip_gradient(grad, max_norm=1.0)
            m[:] = beta1 * m + (1 - beta1) * grad
            v[:] = beta2 * v + (1 - beta2) * (grad ** 2)
            m_hat = m / (1 - beta1 ** t)
            v_hat = v / (1 - beta2 ** t)
            p -= current_lr * m_hat / (np.sqrt(v_hat) + epsilon)

        if batchidx % 100 == 0:
            acc = np.mean(np.argmax(prob_softmax, axis=1) == y_batch)
            last_attn = attW[0, 0, -1, :]
            print(f"Epoch {epoch+1}, Batch {batchidx}, Loss: {loss:.4f}, Acc: {acc:.4f}, attn: {last_attn.max()}")
            print(f"  Attention: {last_attn.round(3)}")

(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
Epoch 1, Batch 0, Loss: 9.7336, Acc: 0.0000, attn: 0.015736014172744166
  Attention: [0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016
 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016
 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016
 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016 0.016
 0.016 0.016 0.016 0.016 0.016 0.016 0.015 0.015 0.015 0.015 0.015 0.016
 0.016 0.016 0.016 0.016]
(32,)


/var/folders/vh/cpmg6tdx2q33v5n2z2c4lcv00000gn/T/ipykernel_1703/1964734869.py:170: RuntimeWarning: divide by zero encountered in matmul
  scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)
/var/folders/vh/cpmg6tdx2q33v5n2z2c4lcv00000gn/T/ipykernel_1703/1964734869.py:170: RuntimeWarning: overflow encountered in matmul
  scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)
/var/folders/vh/cpmg6tdx2q33v5n2z2c4lcv00000gn/T/ipykernel_1703/1964734869.py:170: RuntimeWarning: invalid value encountered in matmul
  scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)
/var/folders/vh/cpmg6tdx2q33v5n2z2c4lcv00000gn/T/ipykernel_1703/1964734869.py:175: RuntimeWarning: divide by zero encountered in matmul
  attOut = np.matmul(attW, V)
/var/folders/vh/cpmg6tdx2q33v5n2z2c4lcv00000gn/T/ipykernel_1703/1964734869.py:175: RuntimeWarning: overflow encountered in matmul
  attOut = np.matmul(attW, V)
/var/folders/vh/cpmg6tdx2q33v5n2z2c4lcv00000gn/T/ipykernel_1703/19

(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64, 64)
bsdfsadf (32, 12131)
(32,)
(32, 8, 64,

KeyboardInterrupt: 

In [ ]:


device = Metal.MTLCreateSystemDefaultDevice()#handler to gpu device is used for anything Metal 
print(f"Metal GPU: {device.name()}")

with open("kernel_claude_test.metal", "r") as f:#Metal kerner 
    kernel_source = f.read()

library, error = device.newLibraryWithSource_options_error_(kernel_source, None, None)#parse the text from the metal file and compile it into a lirbary of gpu functions
if library is None:
    raise RuntimeError(f"Failed to compile Metal kernel: {error}")
print("Metal kernel compiled successfully.")

command_queue = device.newCommandQueue()#used to submit work to the gpu and it executes the submits in order and handles synchronization

_NAMES = [
    "gpu_dot", "gpu_batched_matmul", "gpu_transpose_2d", "gpu_transpose_4d",
    "gpu_embedding_lookup", "gpu_gelu", "gpu_gelu_backward",
    "gpu_softmax", "gpu_softmax_4d", "gpu_softmax_backward_4d",
    "gpu_layer_norm_forward", "gpu_layer_norm_backward_dx", "gpu_layer_norm_backward_gamma_beta",
    "gpu_element_add", "gpu_scalar_div",
    "gpu_bias_add_3d", "gpu_bias_add_2d",
    "gpu_sum_3d_to_1d", "gpu_sum_2d_to_1d",
    "gpu_cross_entropy_loss", "gpu_softmax_ce_backward",
    "gpu_scatter_add", "gpu_zero_buffer", "gpu_adam_update","gpu_scale"
]#.metal names 

P = {}
for name in _NAMES:
    fn = library.newFunctionWithName_(name)
    if fn is None:
        print(f"Missing: {name}")
        continue
    pipe, err = device.newComputePipelineStateWithFunction_error_(fn, None)#makes pipeline for each function in .metal file this contains information about registers it needs , threadgroup memory , and max size of threadgroup 
    P[name] = pipe #stores all function pipelines 

print(f"Loaded {len(P)} pipelines")

#for a 4*4 matrix we have 16 threads one for every output value threads are individual workers that execute simutenuesly thread 0 is out[0][0], thread 1 is out[0][1], thread 2 is out[0][2] ... thread 15 is out[3][3] if we have 8 registers(row col sum k temps) each thread occupies 8 physical register slots so in this case it would be 128 slots (16*8) each gpu core has 4096 slots 


class Gbuffer:
    __slots__ = ('buf', 'shape', 'size')#slots tells python Gbuffer class only has 3 atributes buf, shape, size this is for memory optimization

    def __init__(self, buf, shape):#initialize object gbuffer with a metal buffer and its shape and size if shape 256, 8 , 64 then size is 256*8*64
        self.buf = buf
        self.shape = tuple(shape)
        self.size = int(np.prod(shape))

    @staticmethod #static means you can only call the method using Gbufer.from_numpy(arr) and not from an instance of Gbuffer
    def from_numpy(arr):
        arr = np.ascontiguousarray(arr, dtype=np.float32)#asscontiguousarray checkss for memory gaps in an array of floats 4 bytes per number 
        buf = device.newBufferWithBytes_length_options_(arr.tobytes(), arr.nbytes, 0)#arr.tobytes is the raw bytes arr.bytes total byte count , newBufferWithBytes_length_options_ allocates metal buffer copies bytes in it the 0 mean both cpu and gpu can access it
        return Gbuffer(buf, arr.shape)

    @staticmethod
    def from_numpy_uint(arr): #used for token id's and labels which are integers and not floats
        arr = np.ascontiguousarray(arr, dtype=np.uint32)
        buf = device.newBufferWithBytes_length_options_(arr.tobytes(), arr.nbytes, 0)
        return Gbuffer(buf, arr.shape)

    @staticmethod
    def zeros(shape): #creates buffer full of zeros 
        size = int(np.prod(shape))
        arr = np.zeros(size, dtype=np.float32)
        buf = device.newBufferWithBytes_length_options_(arr.tobytes(), arr.nbytes, 0)
        return Gbuffer(buf, shape)

    @staticmethod
    def alloc(shape):#for memory allocation and placement of buffers on gpu without initializing with data
        size = int(np.prod(shape))
        buf = device.newBufferWithLength_options_(size * 4, 0)# times 4 bytes per float32 number
        return Gbuffer(buf, shape)

    def write_numpy(self, arr):#overwrite the contents of the buffer with new data from a numpy array this is used for updating weights and gradients during training to not overfill with new buffers and cause memory fragmentation
        arr = np.ascontiguousarray(arr, dtype=np.float32)
        data = arr.tobytes()
        self.buf.contents().as_buffer(len(data))[:] = data#gets a pointer to the memory and make sit a python writable buffer and copy the data in bytes used in training loop in X_batch and y_batch 

    def write_numpy_uint(self, arr):
        arr = np.ascontiguousarray(arr, dtype=np.uint32)
        data = arr.tobytes()
        self.buf.contents().as_buffer(len(data))[:] = data

    def numpy(self):
        return np.frombuffer(
            self.buf.contents().as_buffer(self.size * 4), dtype=np.float32
        ).copy().reshape(self.shape)#read data from gpu and converts it back to numpy array 

    def numpy_uint(self):
        return np.frombuffer(
            self.buf.contents().as_buffer(self.size * 4), dtype=np.uint32
        ).copy().reshape(self.shape)

    def reshape(self, new_shape):#reshapes arrays same bytes in memory but different shape for example 256,8,64 to 512,64 or 256,512 to 256,8,64 this is used for reshaping in attention and layer norm
        new_shape = list(new_shape)
        if -1 in new_shape:
            idx = new_shape.index(-1)
            known = 1
            for i, s in enumerate(new_shape):
                if i != idx:
                    known *= s
            new_shape[idx] = self.size // known
        return Gbuffer(self.buf, tuple(new_shape))



#empty dictionaries for storing previous scalar buffers so we dont have to create new buffers for the same scalar values and waste memory on gpu
_uint_cache = {}
_float_cache = {}
#create metal buffers holding one uint32 value for passing dimensions(MNK), First time its been called creates a 4 byte buffer containing a number and when we reuse it it return the same buffer no new allocation needed , forward backward calls _u a lot so insted of creating new buffers always we just reuse already created ones 
def _u(v):
    v = int(v)
    if v not in _uint_cache:
        _uint_cache[v] = device.newBufferWithBytes_length_options_(
            np.array([v], dtype=np.uint32).tobytes(), 4, 0)
    return _uint_cache[v]

def _f(v):
    v = float(v)
    if v not in _float_cache:
        _float_cache[v] = device.newBufferWithBytes_length_options_(
            np.array([v], dtype=np.float32).tobytes(), 4, 0)
    return _float_cache[v]



#this is the to do list what that means is taht nothing executes until submit() is called 
class CommandStream:
    def __init__(self):
        self.cb = command_queue.commandBuffer()
#enc is the command encoder which is used to encode commands for the gpu to execute its like a builder for the command buffer you set the pipeline state and buffers and then dispatch threads and then end encoding when done
    def _enc(self, pipeline_name, buffers, grid, group):
        encoder = self.cb.computeCommandEncoder()
        encoder.setComputePipelineState_(P[pipeline_name])
        for i, b in enumerate(buffers):
            if isinstance(b, Gbuffer):
                encoder.setBuffer_offset_atIndex_(b.buf, 0, i)
            else:
                encoder.setBuffer_offset_atIndex_(b, 0, i)
        encoder.dispatchThreads_threadsPerThreadgroup_(
            Metal.MTLSize(*grid), Metal.MTLSize(*group))
        encoder.endEncoding()
#
    def submit(self):
        self.cb.commit()
        self.cb.waitUntilCompleted()
        err = self.cb.error()
        if err:
            print(f"GPU ERROR: {err.localizedDescription()}")
        self.cb = command_queue.commandBuffer()

    def dot(self, a, b, out):
        M, K = a.shape; _, N = b.shape
        self._enc("gpu_dot", [a, b, out, _u(M), _u(K), _u(N)],
                  (N, M, 1), (min(N, 16), min(M, 16), 1))

    def dot_3d(self, a, b, out):
        batch, seq, K = a.shape; _, N = b.shape
        self.dot(a.reshape((batch * seq, K)), b, out.reshape((batch * seq, N)))

    def transpose_2d(self, a, out):
        M, N = a.shape
        self._enc("gpu_transpose_2d", [a, out, _u(M), _u(N)],
                  (N, M, 1), (min(N, 16), min(M, 16), 1))

    def transpose_4d(self, a, out, swap):
        d0, d1, d2, d3 = a.shape
        total = a.size
        mode = 0 if swap in ((1, 2), (2, 1)) else 1
        self._enc("gpu_transpose_4d",
                  [a, out, _u(d0), _u(d1), _u(d2), _u(d3), _u(mode)],
                  (total, 1, 1), (min(total, 256), 1, 1))

    def embedding_lookup(self, embed, indices, out):
        vocab_size, embed_dim = embed.shape
        if len(indices.shape) == 1:
            num_pos = indices.shape[0]
        else:
            batch, seq = indices.shape
            num_pos = batch * seq
        idx_flat = indices.reshape((num_pos,))
        out_flat = out.reshape((num_pos, embed_dim))
        self._enc("gpu_embedding_lookup",
                  [embed, idx_flat, out_flat, _u(num_pos), _u(embed_dim)],
                  (embed_dim, num_pos, 1), (min(embed_dim, 64), min(num_pos, 4), 1))

    def element_add(self, a, b, out):
        self._enc("gpu_element_add", [a, b, out],
                  (a.size, 1, 1), (min(a.size, 256), 1, 1))

    def scalar_div(self, a, scalar, out):
        self._enc("gpu_scalar_div", [a, out, _f(float(scalar))],
                  (a.size, 1, 1), (min(a.size, 256), 1, 1))

    def softmax_4d(self, a, out):
        bs, nh, s1, s2 = a.shape
        self._enc("gpu_softmax_4d",
                  [a, out, _u(bs), _u(nh), _u(s1), _u(s2)],
                  (bs, nh, s1), (min(bs, 4), min(nh, 4), min(s1, 16)))

    def batched_matmul(self, a, b, out):
        bs, nh, M, K = a.shape; _, _, _, N = b.shape
        self._enc("gpu_batched_matmul",
                  [a, b, out, _u(bs), _u(nh), _u(M), _u(K), _u(N)],
                  (N, M, bs * nh), (min(N, 8), min(M, 8), min(bs * nh, 4)))

    def layer_norm_forward(self, x, gamma, beta, out, x_norm, variance, eps=1e-5):
        if len(x.shape) == 3:
            b, s, d = x.shape
            x2 = x.reshape((b * s, d))
            o2 = out.reshape((b * s, d))
            n2 = x_norm.reshape((b * s, d))
        else:
            x2 = x; o2 = out; n2 = x_norm
        rows, cols = x2.shape
        self._enc("gpu_layer_norm_forward",
                  [x2, gamma, beta, o2, n2, variance,
                   _u(rows), _u(cols), _f(eps)],
                  (rows, 1, 1), (min(rows, 256), 1, 1))

    def layer_norm_backward_dx(self, dout, x_norm, variance, gamma, dx, eps=1e-5):
        if len(dout.shape) == 3:
            b, s, d = dout.shape
            d2 = dout.reshape((b * s, d))
            n2 = x_norm.reshape((b * s, d))
            x2 = dx.reshape((b * s, d))
        else:
            d2 = dout; n2 = x_norm; x2 = dx
        rows, cols = d2.shape
        self._enc("gpu_layer_norm_backward_dx",
                  [d2, n2, variance, gamma, x2,
                   _u(rows), _u(cols), _f(eps)],
                  (rows, 1, 1), (min(rows, 256), 1, 1))

    def layer_norm_backward_gamma_beta(self, dout, x_norm, dgamma, dbeta):
        if len(dout.shape) == 3:
            b, s, d = dout.shape
            d2 = dout.reshape((b * s, d))
            n2 = x_norm.reshape((b * s, d))
        else:
            d2 = dout; n2 = x_norm
        rows, cols = d2.shape
        self._enc("gpu_layer_norm_backward_gamma_beta",
                  [d2, n2, dgamma, dbeta, _u(rows), _u(cols)],
                  (cols, 1, 1), (min(cols, 256), 1, 1))

    def gelu(self, a, out):
        self._enc("gpu_gelu", [a, out],
                  (a.size, 1, 1), (min(a.size, 256), 1, 1))

    def gelu_backward(self, input_buf, grad_out, out):
        self._enc("gpu_gelu_backward", [input_buf, grad_out, out],
                  (input_buf.size, 1, 1), (min(input_buf.size, 256), 1, 1))

    def bias_add_3d(self, a, bias, out):
        batch, seq, dim = a.shape
        self._enc("gpu_bias_add_3d",
                  [a, bias, out, _u(batch), _u(seq), _u(dim)],
                  (dim, seq, batch), (min(dim, 16), min(seq, 8), min(batch, 4)))

    def bias_add_2d(self, a, bias, out):
        rows, cols = a.shape
        self._enc("gpu_bias_add_2d", [a, bias, out, _u(rows), _u(cols)],
                  (cols, rows, 1), (min(cols, 16), min(rows, 16), 1))

    def softmax(self, a, out):
        rows, cols = a.shape
        self._enc("gpu_softmax", [a, out, _u(rows), _u(cols)],
                  (rows, 1, 1), (min(rows, 256), 1, 1))

    def cross_entropy_loss(self, probs, labels, out):
        bs, nc = probs.shape
        self._enc("gpu_cross_entropy_loss",
                  [probs, labels, out, _u(bs), _u(nc)],
                  (bs, 1, 1), (min(bs, 256), 1, 1))

    def softmax_ce_backward(self, probs, labels, num_predictions, out):
        bs, nc = probs.shape
        scale = 1.0 / num_predictions
        self._enc("gpu_softmax_ce_backward",
                  [probs, labels, out, _u(bs), _u(nc), _f(scale)],
                  (nc, bs, 1), (min(nc, 16), min(bs, 16), 1))

    def sum_2d_to_1d(self, a, out):
        rows, cols = a.shape
        self._enc("gpu_sum_2d_to_1d", [a, out, _u(rows), _u(cols)],
                  (cols, 1, 1), (min(cols, 256), 1, 1))

    def sum_3d_to_1d(self, a, out):
        batch, seq, dim = a.shape
        self._enc("gpu_sum_3d_to_1d",
                  [a, out, _u(batch), _u(seq), _u(dim)],
                  (dim, 1, 1), (min(dim, 256), 1, 1))

    def softmax_backward_4d(self, attW, dattW, out):
        d0, d1, d2, d3 = attW.shape
        self._enc("gpu_softmax_backward_4d",
                  [attW, dattW, out, _u(d0), _u(d1), _u(d2), _u(d3)],
                  (d3, d2, d0 * d1), (min(d3, 8), min(d2, 8), min(d0 * d1, 4)))

    def scatter_add(self, out, indices, updates):
        """out must be zeroed before calling."""
        vocab_size, embed_dim = out.shape
        num_indices = indices.size
        updates_2d = updates.reshape((num_indices, embed_dim))
        self._enc("gpu_scatter_add",
                  [out, indices, updates_2d, _u(num_indices), _u(embed_dim)],
                  (embed_dim, num_indices, 1), (min(embed_dim, 64), min(num_indices, 4), 1))

    def zero_buffer(self, buf):
        self._enc("gpu_zero_buffer", [buf],
                  (buf.size, 1, 1), (min(buf.size, 256), 1, 1))

    def adam_update(self, param, grad, m, v, lr, beta1, beta2, epsilon, t):
        total = param.size
        beta1_t = beta1 ** t
        beta2_t = beta2 ** t
        self._enc("gpu_adam_update",
                  [param, grad, m, v,
                   _f(lr), _f(beta1), _f(beta2), _f(epsilon),
                   _f(beta1_t), _f(beta2_t)],
                  (total, 1, 1), (min(total, 256), 1, 1))




def positional_encoding(block_size, embed_dim):
    pos_enc = np.zeros((block_size, embed_dim), dtype=np.float32)
    for pos in range(block_size):
        for i in range(0, embed_dim, 2):
            pos_enc[pos, i] = np.sin(pos / (10000 ** (i / embed_dim)))
            if i + 1 < embed_dim:
                pos_enc[pos, i + 1] = np.cos(pos / (10000 ** (i / embed_dim)))
    return pos_enc



print("GPU-resident batched ops ready.")
 

Metal GPU: Apple M4
Metal kernel compiled successfully.
Loaded 25 pipelines
GPU-resident batched ops ready.


In [ ]:
#Claude help 



n_blocks = 4
epochs = 20
block_size = 64
embed_dim = 128
hidden_size = 1024
batch_size = 128
num_heads = 8
head_dim = embed_dim // num_heads
num_samples = X_train.shape[0]
steps_per_epoch = num_samples // batch_size
lr = 3e-4 
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
t_step = 0

# Shorthand
BS = batch_size
SQ = block_size
ED = embed_dim
HD = hidden_size
NH = num_heads
HE = head_dim
VS = vocab_size
FLAT = BS * SQ

sqrt_head_dim = float(np.sqrt(head_dim))



W_embed = Gbuffer.from_numpy(np.random.randn(VS, ED).astype(np.float32) * 0.01)
W_out = Gbuffer.from_numpy(np.random.randn(ED, VS).astype(np.float32) * 0.01)
b_out = Gbuffer.zeros((VS,))

m_W_embed, v_W_embed = Gbuffer.zeros(W_embed.shape), Gbuffer.zeros(W_embed.shape)
m_W_out, v_W_out = Gbuffer.zeros(W_out.shape), Gbuffer.zeros(W_out.shape)
m_b_out, v_b_out = Gbuffer.zeros(b_out.shape), Gbuffer.zeros(b_out.shape)

block_weights = []
block_adam = []
for _ in range(n_blocks):
    w = {
        'ln1_gamma': Gbuffer.from_numpy(np.ones(ED, dtype=np.float32)),
        'ln1_beta': Gbuffer.zeros((ED,)),
        'W_q': Gbuffer.from_numpy(np.random.randn(ED, ED).astype(np.float32) * 0.02),
        'W_k': Gbuffer.from_numpy(np.random.randn(ED, ED).astype(np.float32) * 0.02),
        'W_v': Gbuffer.from_numpy(np.random.randn(ED, ED).astype(np.float32) * 0.02),
        'W_o': Gbuffer.from_numpy(np.random.randn(ED, ED).astype(np.float32) * 0.02),
        'ln2_gamma': Gbuffer.from_numpy(np.ones(ED, dtype=np.float32)),
        'ln2_beta': Gbuffer.zeros((ED,)),
        'W_ff1': Gbuffer.from_numpy(np.random.randn(ED, HD).astype(np.float32) * np.sqrt(2 / ED)),
        'b_ff1': Gbuffer.zeros((HD,)),
        'W_ff2': Gbuffer.from_numpy(np.random.randn(HD, ED).astype(np.float32) * np.sqrt(2 / HD)),
        'b_ff2': Gbuffer.zeros((ED,)),
    }
    block_weights.append(w)
    a = {}
    for key in w:
        a['m_' + key] = Gbuffer.zeros(w[key].shape)
        a['v_' + key] = Gbuffer.zeros(w[key].shape)
    block_adam.append(a)



CAUSAL_MASK = Gbuffer.from_numpy(
    np.broadcast_to(
        np.triu(np.ones((SQ, SQ), dtype=np.float32), k=1) * -1e5,
        (BS, NH, SQ, SQ)).copy())

PE = Gbuffer.from_numpy(
    np.broadcast_to(positional_encoding(SQ, ED),
                    (BS, SQ, ED)).copy())


# Input buffers
X_batch = Gbuffer.alloc((BS, SQ))
y_batch = Gbuffer.alloc((BS, SQ))
y_flat = y_batch.reshape((FLAT,))

# Embedding
embedded = Gbuffer.alloc((BS, SQ, ED))
embedded_pe = Gbuffer.alloc((BS, SQ, ED))

# Per-block forward cache 
fwd = []

warmup_steps = 2000
total_steps = epochs * steps_per_epoch

def get_lr(step):
    if step < warmup_steps:
        return lr * step / warmup_steps
    # Cosine decay after warmup
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return lr * 0.5 * (1 + np.cos(np.pi * progress))
for _ in range(n_blocks):
    fwd.append({
        'ln1_out':    Gbuffer.alloc((BS, SQ, ED)),
        'ln1_xnorm':  Gbuffer.alloc((BS, SQ, ED)),
        'ln1_var':     Gbuffer.alloc((FLAT,)),
        'Qpre':        Gbuffer.alloc((BS, SQ, ED)),
        'Kpre':        Gbuffer.alloc((BS, SQ, ED)),
        'Vpre':        Gbuffer.alloc((BS, SQ, ED)),
        'Q':           Gbuffer.alloc((BS, NH, SQ, HE)),
        'K':           Gbuffer.alloc((BS, NH, SQ, HE)),
        'V':           Gbuffer.alloc((BS, NH, SQ, HE)),
        'attW':        Gbuffer.alloc((BS, NH, SQ, SQ)),
        'attOut_res':  Gbuffer.alloc((BS, SQ, ED)),  # attOut after transpose back
        'res1':        Gbuffer.alloc((BS, SQ, ED)),
        'ln2_out':     Gbuffer.alloc((BS, SQ, ED)),
        'ln2_xnorm':   Gbuffer.alloc((BS, SQ, ED)),
        'ln2_var':      Gbuffer.alloc((FLAT,)),
        'ffn_hidden':  Gbuffer.alloc((BS, SQ, HD)),
        'ffn_gelu':    Gbuffer.alloc((BS, SQ, HD)),
        'res2':        Gbuffer.alloc((BS, SQ, ED)),
    })

# Forward (reused across blocks)
fsc = {
    'K_T':        Gbuffer.alloc((BS, NH, HE, SQ)),
    'scores':     Gbuffer.alloc((BS, NH, SQ, SQ)),
    'scores_sc':  Gbuffer.alloc((BS, NH, SQ, SQ)),
    'scores_mk':  Gbuffer.alloc((BS, NH, SQ, SQ)),
    'attOut':     Gbuffer.alloc((BS, NH, SQ, HE)),
    'projOut':    Gbuffer.alloc((BS, SQ, ED)),
    'ffn_z1':     Gbuffer.alloc((BS, SQ, HD)),
    'ffn_z2':     Gbuffer.alloc((BS, SQ, ED)),
    'ffn_out':    Gbuffer.alloc((BS, SQ, ED)),
}

# Output 
logits = Gbuffer.alloc((FLAT, VS))
logits_bias = Gbuffer.alloc((FLAT, VS))
probs = Gbuffer.alloc((FLAT, VS))
losses = Gbuffer.alloc((FLAT,))

# Backward output 
dlogits = Gbuffer.alloc((FLAT, VS))
x_flat_T = Gbuffer.alloc((ED, FLAT))
dW_out_buf = Gbuffer.alloc((ED, VS))
db_out_buf = Gbuffer.alloc((VS,))
W_out_T = Gbuffer.alloc((VS, ED))
dx_flat = Gbuffer.alloc((FLAT, ED))

# Backward 
bk = {
    'gelu_flat_T':     Gbuffer.alloc((HD, FLAT)),
    'dW_ff2':          Gbuffer.alloc((HD, ED)),
    'db_ff2':          Gbuffer.alloc((ED,)),
    'W_ff2_T':         Gbuffer.alloc((ED, HD)),
    'dffn_gelu':       Gbuffer.alloc((BS, SQ, HD)),
    'dffn_hidden':     Gbuffer.alloc((BS, SQ, HD)),
    'ln2_flat_T':      Gbuffer.alloc((ED, FLAT)),
    'dW_ff1':          Gbuffer.alloc((ED, HD)),
    'db_ff1':          Gbuffer.alloc((HD,)),
    'W_ff1_T':         Gbuffer.alloc((HD, ED)),
    'dln2_out':        Gbuffer.alloc((BS, SQ, ED)),
    'dres1_ln2':       Gbuffer.alloc((BS, SQ, ED)),
    'dln2_gamma':      Gbuffer.alloc((ED,)),
    'dln2_beta':       Gbuffer.alloc((ED,)),
    'dres1':           Gbuffer.alloc((BS, SQ, ED)),
    'attOut_flat_T':   Gbuffer.alloc((ED, FLAT)),
    'dW_o':            Gbuffer.alloc((ED, ED)),
    'W_o_T':           Gbuffer.alloc((ED, ED)),
    'd_att_comb':      Gbuffer.alloc((BS, SQ, ED)),
    'd_attOut':        Gbuffer.alloc((BS, NH, SQ, HE)),
    'V_T':             Gbuffer.alloc((BS, NH, HE, SQ)),
    'dattW':           Gbuffer.alloc((BS, NH, SQ, SQ)),
    'attW_T':          Gbuffer.alloc((BS, NH, SQ, SQ)),
    'dV_heads':        Gbuffer.alloc((BS, NH, SQ, HE)),
    'd_scores':        Gbuffer.alloc((BS, NH, SQ, SQ)),
    'd_scores_sc':     Gbuffer.alloc((BS, NH, SQ, SQ)),
    'dQ_heads':        Gbuffer.alloc((BS, NH, SQ, HE)),
    'd_scores_T':      Gbuffer.alloc((BS, NH, SQ, SQ)),
    'dK_heads':        Gbuffer.alloc((BS, NH, SQ, HE)),
    'dQ_pre':          Gbuffer.alloc((BS, SQ, NH, HE)),
    'dK_pre':          Gbuffer.alloc((BS, SQ, NH, HE)),
    'dV_pre':          Gbuffer.alloc((BS, SQ, NH, HE)),
    'ln1_flat_T':      Gbuffer.alloc((ED, FLAT)),
    'dW_q':            Gbuffer.alloc((ED, ED)),
    'dW_k':            Gbuffer.alloc((ED, ED)),
    'dW_v':            Gbuffer.alloc((ED, ED)),
    'W_q_T':           Gbuffer.alloc((ED, ED)),
    'W_k_T':           Gbuffer.alloc((ED, ED)),
    'W_v_T':           Gbuffer.alloc((ED, ED)),
    'dln1_q':          Gbuffer.alloc((BS, SQ, ED)),
    'dln1_k':          Gbuffer.alloc((BS, SQ, ED)),
    'dln1_v':          Gbuffer.alloc((BS, SQ, ED)),
    'dln1_qk':         Gbuffer.alloc((BS, SQ, ED)),
    'dln1_qkv':        Gbuffer.alloc((BS, SQ, ED)),
    'dx_ln1':          Gbuffer.alloc((BS, SQ, ED)),
    'dln1_gamma':      Gbuffer.alloc((ED,)),
    'dln1_beta':       Gbuffer.alloc((ED,)),
    'dx':              Gbuffer.alloc((BS, SQ, ED)),
}

# Per-block gradient storage 
block_grads = []
for _ in range(n_blocks):
    block_grads.append({
        'dW_q':       Gbuffer.alloc((ED, ED)),
        'dW_k':       Gbuffer.alloc((ED, ED)),
        'dW_v':       Gbuffer.alloc((ED, ED)),
        'dW_o':       Gbuffer.alloc((ED, ED)),
        'dln1_gamma': Gbuffer.alloc((ED,)),
        'dln1_beta':  Gbuffer.alloc((ED,)),
        'dW_ff1':     Gbuffer.alloc((ED, HD)),
        'db_ff1':     Gbuffer.alloc((HD,)),
        'dW_ff2':     Gbuffer.alloc((HD, ED)),
        'db_ff2':     Gbuffer.alloc((ED,)),
        'dln2_gamma': Gbuffer.alloc((ED,)),
        'dln2_beta':  Gbuffer.alloc((ED,)),
    })

# Embedding gradient
dW_embed = Gbuffer.alloc((VS, ED))

print(f"Model: {n_blocks} blocks, {ED}d, {NH} heads, {HD} FFN")
print(f"Training: {num_samples:,} samples, {steps_per_epoch:,} batches/epoch, bs={BS}")
print("All buffers pre-allocated. Zero allocations during training.")
print("=" * 60)



for epoch in range(epochs):
    perm = np.random.permutation(num_samples)
    X_sh = X_train[perm]
    y_sh = y_train[perm]
    epoch_start = time.time()

    for batchidx in range(steps_per_epoch):
        with autorelease_pool():
            si = batchidx * BS
            ei = si + BS

            X_batch.write_numpy_uint(X_sh[si:ei].astype(np.uint32))
            y_batch.write_numpy_uint(y_sh[si:ei].astype(np.uint32))

            s = CommandStream()

            # forward pass

            s.embedding_lookup(W_embed, X_batch, embedded)
            s.element_add(embedded, PE, embedded_pe)

            for blk in range(n_blocks):
                w = block_weights[blk]
                f = fwd[blk]
                x_in = embedded_pe if blk == 0 else fwd[blk - 1]['res2']

                # LayerNorm 1
                s.layer_norm_forward(x_in, w['ln1_gamma'], w['ln1_beta'],
                                    f['ln1_out'], f['ln1_xnorm'], f['ln1_var'])

                # Q K V projections
                s.dot_3d(f['ln1_out'], w['W_q'], f['Qpre'])
                s.transpose_4d(f['Qpre'].reshape((BS, SQ, NH, HE)), f['Q'], (1, 2))

                s.dot_3d(f['ln1_out'], w['W_k'], f['Kpre'])
                s.transpose_4d(f['Kpre'].reshape((BS, SQ, NH, HE)), f['K'], (1, 2))

                s.dot_3d(f['ln1_out'], w['W_v'], f['Vpre'])
                s.transpose_4d(f['Vpre'].reshape((BS, SQ, NH, HE)), f['V'], (1, 2))

                # Attention scores
                s.transpose_4d(f['K'], fsc['K_T'], (2, 3))
                s.batched_matmul(f['Q'], fsc['K_T'], fsc['scores'])
                s.scalar_div(fsc['scores'], sqrt_head_dim, fsc['scores_sc'])
                s.element_add(fsc['scores_sc'], CAUSAL_MASK, fsc['scores_mk'])
                s.softmax_4d(fsc['scores_mk'], f['attW'])

                # Attention output
                s.batched_matmul(f['attW'], f['V'], fsc['attOut'])
                s.transpose_4d(fsc['attOut'], f['attOut_res'].reshape((BS, SQ, NH, HE)), (1, 2))
                s.dot_3d(f['attOut_res'], w['W_o'], fsc['projOut'])

                # Residual 1
                s.element_add(x_in, fsc['projOut'], f['res1'])

                # LayerNorm 2
                s.layer_norm_forward(f['res1'], w['ln2_gamma'], w['ln2_beta'],
                                    f['ln2_out'], f['ln2_xnorm'], f['ln2_var'])

                # FFN
                s.dot_3d(f['ln2_out'], w['W_ff1'], fsc['ffn_z1'])
                s.bias_add_3d(fsc['ffn_z1'], w['b_ff1'], f['ffn_hidden'])
                s.gelu(f['ffn_hidden'], f['ffn_gelu'])
                s.dot_3d(f['ffn_gelu'], w['W_ff2'], fsc['ffn_z2'])
                s.bias_add_3d(fsc['ffn_z2'], w['b_ff2'], fsc['ffn_out'])

                # Residual 2
                s.element_add(f['res1'], fsc['ffn_out'], f['res2'])

            # Output projection
            x_final = fwd[n_blocks - 1]['res2'].reshape((FLAT, ED))
            s.dot(x_final, W_out, logits)
            s.bias_add_2d(logits, b_out, logits_bias)
            s.softmax(logits_bias, probs)
            s.cross_entropy_loss(probs, y_flat, losses)

            #backward pass

            s.softmax_ce_backward(probs, y_flat, FLAT, dlogits)

            # Output layer grads
            s.transpose_2d(x_final, x_flat_T)
            s.dot(x_flat_T, dlogits, dW_out_buf)
            s.sum_2d_to_1d(dlogits, db_out_buf)
            s.transpose_2d(W_out, W_out_T)
            s.dot(dlogits, W_out_T, dx_flat)

            for blk in reversed(range(n_blocks)):
                
                w = block_weights[blk]
                f = fwd[blk]
                bg = block_grads[blk]
                x_in = embedded_pe if blk == 0 else fwd[blk - 1]['res2']

                # Current dx
                if blk == n_blocks - 1:
                    cur_dx = dx_flat.reshape((BS, SQ, ED))
                else:
                    cur_dx = bk['dx']

                # FFN backward 
                s.transpose_2d(f['ffn_gelu'].reshape((FLAT, HD)), bk['gelu_flat_T'])
                s.dot(bk['gelu_flat_T'], cur_dx.reshape((FLAT, ED)), bg['dW_ff2'])
                s.sum_3d_to_1d(cur_dx, bg['db_ff2'])
                s.transpose_2d(w['W_ff2'], bk['W_ff2_T'])
                s.dot_3d(cur_dx, bk['W_ff2_T'], bk['dffn_gelu'])
                s.gelu_backward(f['ffn_hidden'], bk['dffn_gelu'], bk['dffn_hidden'])

                s.transpose_2d(f['ln2_out'].reshape((FLAT, ED)), bk['ln2_flat_T'])
                s.dot(bk['ln2_flat_T'], bk['dffn_hidden'].reshape((FLAT, HD)), bg['dW_ff1'])
                s.sum_3d_to_1d(bk['dffn_hidden'], bg['db_ff1'])
                s.transpose_2d(w['W_ff1'], bk['W_ff1_T'])
                s.dot_3d(bk['dffn_hidden'], bk['W_ff1_T'], bk['dln2_out'])

                # LN2 backward
                s.layer_norm_backward_dx(bk['dln2_out'], f['ln2_xnorm'], f['ln2_var'],
                                        w['ln2_gamma'], bk['dres1_ln2'])
                s.layer_norm_backward_gamma_beta(bk['dln2_out'], f['ln2_xnorm'],
                                                bg['dln2_gamma'], bg['dln2_beta'])
                s.element_add(cur_dx, bk['dres1_ln2'], bk['dres1'])

                #Attention backward 
                s.transpose_2d(f['attOut_res'].reshape((FLAT, ED)), bk['attOut_flat_T'])
                s.dot(bk['attOut_flat_T'], bk['dres1'].reshape((FLAT, ED)), bg['dW_o'])
                s.transpose_2d(w['W_o'], bk['W_o_T'])
                s.dot_3d(bk['dres1'], bk['W_o_T'], bk['d_att_comb'])

                s.transpose_4d(bk['d_att_comb'].reshape((BS, SQ, NH, HE)),
                            bk['d_attOut'], (1, 2))

                s.transpose_4d(f['V'], bk['V_T'], (2, 3))
                s.batched_matmul(bk['d_attOut'], bk['V_T'], bk['dattW'])
                s.transpose_4d(f['attW'], bk['attW_T'], (2, 3))
                s.batched_matmul(bk['attW_T'], bk['d_attOut'], bk['dV_heads'])

                s.softmax_backward_4d(f['attW'], bk['dattW'], bk['d_scores'])
                s.scalar_div(bk['d_scores'], sqrt_head_dim, bk['d_scores_sc'])

                s.batched_matmul(bk['d_scores_sc'], f['K'], bk['dQ_heads'])
                s.transpose_4d(bk['d_scores_sc'], bk['d_scores_T'], (2, 3))
                s.batched_matmul(bk['d_scores_T'], f['Q'], bk['dK_heads'])

                s.transpose_4d(bk['dQ_heads'], bk['dQ_pre'], (1, 2))
                s.transpose_4d(bk['dK_heads'], bk['dK_pre'], (1, 2))
                s.transpose_4d(bk['dV_heads'], bk['dV_pre'], (1, 2))

                dQ_res = bk['dQ_pre'].reshape((BS, SQ, ED))
                dK_res = bk['dK_pre'].reshape((BS, SQ, ED))
                dV_res = bk['dV_pre'].reshape((BS, SQ, ED))

                s.transpose_2d(f['ln1_out'].reshape((FLAT, ED)), bk['ln1_flat_T'])
                s.dot(bk['ln1_flat_T'], dQ_res.reshape((FLAT, ED)), bg['dW_q'])
                s.dot(bk['ln1_flat_T'], dK_res.reshape((FLAT, ED)), bg['dW_k'])
                s.dot(bk['ln1_flat_T'], dV_res.reshape((FLAT, ED)), bg['dW_v'])

                s.transpose_2d(w['W_q'], bk['W_q_T'])
                s.dot_3d(dQ_res, bk['W_q_T'], bk['dln1_q'])
                s.transpose_2d(w['W_k'], bk['W_k_T'])
                s.dot_3d(dK_res, bk['W_k_T'], bk['dln1_k'])
                s.transpose_2d(w['W_v'], bk['W_v_T'])
                s.dot_3d(dV_res, bk['W_v_T'], bk['dln1_v'])

                s.element_add(bk['dln1_q'], bk['dln1_k'], bk['dln1_qk'])
                s.element_add(bk['dln1_qk'], bk['dln1_v'], bk['dln1_qkv'])

                # LN1 backward
                s.layer_norm_backward_dx(bk['dln1_qkv'], f['ln1_xnorm'], f['ln1_var'],
                                        w['ln1_gamma'], bk['dx_ln1'])
                s.layer_norm_backward_gamma_beta(bk['dln1_qkv'], f['ln1_xnorm'],
                                                bg['dln1_gamma'], bg['dln1_beta'])
                s.element_add(bk['dres1'], bk['dx_ln1'], bk['dx'])

            # Embedding gradient
            X_flat_idx = X_batch.reshape((FLAT,))
            s.zero_buffer(dW_embed)
            s.scatter_add(dW_embed, X_flat_idx, bk['dx'])

            def clip_grad_inplace(s, grad, max_norm=1.0):
                arr = grad.numpy()
                norm = np.sqrt(np.sum(arr * arr))
                if norm > max_norm:
                    scale = max_norm / norm
                    s._enc("gpu_scale", [grad, _f(scale)],
                           (grad.size, 1, 1), (min(grad.size, 256), 1, 1))

            # Need to submit first so gradients are computed before we read them
            s.submit()
            s =CommandStream()
            # Clip all gradients
            clip_grad_inplace(s, dW_embed)
            clip_grad_inplace(s, dW_out_buf)
            clip_grad_inplace(s, db_out_buf)
            for blk in range(n_blocks):
                bg = block_grads[blk]
                for key in bg:
                    clip_grad_inplace(s, bg[key])

            
            
            # adam update for all parameters
            t_step += 1
            current_lr = get_lr(t_step)

            s.adam_update(W_embed, dW_embed, m_W_embed, v_W_embed,
                        current_lr, beta1, beta2, epsilon, t_step)
            s.adam_update(W_out, dW_out_buf, m_W_out, v_W_out,
                        current_lr, beta1, beta2, epsilon, t_step)
            s.adam_update(b_out, db_out_buf, m_b_out, v_b_out,
                        current_lr, beta1, beta2, epsilon, t_step)

            for blk in range(n_blocks):
                w = block_weights[blk]
                a = block_adam[blk]
                bg = block_grads[blk]
                for key in ['W_q', 'W_k', 'W_v', 'W_o', 'ln1_gamma', 'ln1_beta',
                            'W_ff1', 'b_ff1', 'W_ff2', 'b_ff2', 'ln2_gamma', 'ln2_beta']:
                    s.adam_update(w[key], bg['d' + key], a['m_' + key], a['v_' + key],
                                current_lr, beta1, beta2, epsilon, t_step)

            # ONE submit for the entire step
            s.submit()

            

            if batchidx % 100 == 0:
                loss_val = float(losses.numpy().mean())
                probs_np = probs.numpy()
                y_np = y_flat.numpy_uint()
                acc = np.mean(np.argmax(probs_np, axis=1) == y_np)
                elapsed = time.time() - epoch_start
                bps = (batchidx + 1) / elapsed if elapsed > 0 else 0
                print(f"Epoch {epoch+1} | Batch {batchidx}/{steps_per_epoch} | "
                    f"Loss: {loss_val:.4f} | Acc: {acc:.4f} | {bps:.1f} b/s")

    epoch_time = time.time() - epoch_start
    print(f"--- Epoch {epoch+1} done in {epoch_time:.1f}s ---\n")

print("Training complete.")

Model: 4 blocks, 128d, 8 heads, 1024 FFN
Training: 137,579 samples, 1,074 batches/epoch, bs=128
All buffers pre-allocated. Zero allocations during training.
Epoch 1 | Batch 0/1074 | Loss: 9.4417 | Acc: 0.0013 | 0.9 b/s


KeyboardInterrupt: 

In [ ]:
#test 



import numpy as np
from collections import Counter

block_size = 64
batch_size = 128

# ============================================
# BPE Tokenizer — built from scratch
# ============================================

class BPETokenizer:
    def __init__(self):
        self.merges = []           # ordered list of merge rules: [('t','h'), ('th','e'), ...]
        self.vocab = {}            # token -> id mapping
        self.id_to_token = {}      # id -> token mapping

    def _get_word_freqs(self, text):
        """Split text into words, then split each word into characters + end marker."""
        word_freqs = {}
        words = text.split()
        for word in words:
            chars = tuple(list(word) + ["</w>"])
            word_freqs[chars] = word_freqs.get(chars, 0) + 1
        return word_freqs

    def _get_pair_counts(self, word_freqs):
        """Count how often each adjacent pair appears across all words."""
        pairs = {}
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                pairs[pair] = pairs.get(pair, 0) + freq
        return pairs

    def _merge_pair(self, word_freqs, pair):
        """Replace every occurrence of pair with the merged token."""
        merged = pair[0] + pair[1]
        new_word_freqs = {}
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        return new_word_freqs

    def train(self, text, target_vocab_size):
        """Learn BPE merges from text until we reach target vocab size."""
        print("Training BPE tokenizer...")
        word_freqs = self._get_word_freqs(text)

        # Initial vocab: every unique character in the data
        all_chars = set()
        for word in word_freqs:
            for ch in word:
                all_chars.add(ch)

        # We also need special tokens
        special_tokens = ["<pad>", "<unk>", "<eos>"]
        initial_size = len(all_chars) + len(special_tokens)
        num_merges = target_vocab_size - initial_size

        print(f"  Initial characters: {len(all_chars)}")
        print(f"  Merges to learn: {num_merges}")

        self.merges = []
        for i in range(num_merges):
            pairs = self._get_pair_counts(word_freqs)
            if not pairs:
                break

            best_pair = max(pairs, key=pairs.get)
            count = pairs[best_pair]

            if count < 2:  # no pair appears more than once, stop
                break

            word_freqs = self._merge_pair(word_freqs, best_pair)
            self.merges.append(best_pair)

            if (i + 1) % 500 == 0:
                print(f"  Merge {i+1}: {best_pair[0]} + {best_pair[1]} → {best_pair[0]+best_pair[1]} (count: {count})")

        # Build the final vocabulary
        # Start with special tokens at known indices
        self.vocab = {}
        for i, tok in enumerate(special_tokens):
            self.vocab[tok] = i

        # Add all characters
        for ch in sorted(all_chars):
            if ch not in self.vocab:
                self.vocab[ch] = len(self.vocab)

        # Add all merged tokens
        for pair in self.merges:
            merged = pair[0] + pair[1]
            if merged not in self.vocab:
                self.vocab[merged] = len(self.vocab)

        self.id_to_token = {v: k for k, v in self.vocab.items()}

        print(f"  Final vocab size: {len(self.vocab)}")
        print(f"  Learned {len(self.merges)} merges")
        return self

    def _tokenize_word(self, word):
        """Apply learned merges to a single word."""
        tokens = list(word) + ["</w>"]

        # Apply each merge rule in order
        for pair in self.merges:
            merged = pair[0] + pair[1]
            i = 0
            new_tokens = []
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        return tokens

    def encode(self, text):
        """Text → list of token IDs."""
        ids = []
        words = text.split()
        unk_id = self.vocab["<unk>"]
        for word in words:
            if word == "<eos>":
                ids.append(self.vocab["<eos>"])
                continue
            tokens = self._tokenize_word(word)
            for tok in tokens:
                ids.append(self.vocab.get(tok, unk_id))
        return ids

    def decode(self, ids):
        """List of token IDs → text."""
        tokens = [self.id_to_token.get(i, "<unk>") for i in ids]
        # Join tokens and remove end-of-word markers, restoring spaces
        text = ""
        for tok in tokens:
            if tok in ("<pad>", "<unk>", "<eos>"):
                text += " " + tok
            elif tok.endswith("</w>"):
                text += tok[:-4] + " "  # remove </w>, add space
            else:
                text += tok
        return text.strip()


# ============================================
# Train the tokenizer on your data
# ============================================

text = " ".join(sample.lower() + " <eos>" for sample in dataset['train']['text'])

tokenizer = BPETokenizer()
tokenizer.train(text, target_vocab_size=8000)
vocab_size = len(tokenizer.vocab)

# See what it learned
print("\nExamples:")
for test in ["the cat sat on the mat", "unfortunately", "bioluminescence"]:
    tokens = tokenizer._tokenize_word(test) if " " not in test else sum([tokenizer._tokenize_word(w) for w in test.split()], [])
    print(f"  '{test}' → {tokens}")

# ============================================
# Encode all data
# ============================================

print("\nEncoding dataset...")
data = np.array(tokenizer.encode(text), dtype=np.int64)
print(f"Total tokens: {len(data):,}")

# ============================================
# Create training pairs
# ============================================

X, y = [], []
for i in range(0, len(data) - block_size, block_size):
    X.append(data[i:i + block_size])
    y.append(data[i + 1:i + block_size + 1])

X = np.array(X)
y = np.array(y)

split_idx = int(len(X) * 0.9)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

num_samples = X_train.shape[0]
steps_per_epoch = num_samples // batch_size

print(f"Train samples: {num_samples:,}")
print(f"Test samples: {len(X_test):,}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Vocab size: {vocab_size}")
print(f"Batch size: {batch_size}")

import numpy as np
from collections import Counter

block_size = 64
batch_size = 128

# ============================================
# BPE Tokenizer — built from scratch
# ============================================

class BPETokenizer:
    def __init__(self):
        self.merges = []           # ordered list of merge rules: [('t','h'), ('th','e'), ...]
        self.vocab = {}            # token -> id mapping
        self.id_to_token = {}      # id -> token mapping

    def _get_word_freqs(self, text):
        """Split text into words, then split each word into characters + end marker."""
        word_freqs = {}
        words = text.split()
        for word in words:
            chars = tuple(list(word) + ["</w>"])
            word_freqs[chars] = word_freqs.get(chars, 0) + 1
        return word_freqs

    def _get_pair_counts(self, word_freqs):
        """Count how often each adjacent pair appears across all words."""
        pairs = {}
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                pairs[pair] = pairs.get(pair, 0) + freq
        return pairs

    def _merge_pair(self, word_freqs, pair):
        """Replace every occurrence of pair with the merged token."""
        merged = pair[0] + pair[1]
        new_word_freqs = {}
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        return new_word_freqs

    def train(self, text, target_vocab_size):
        """Learn BPE merges from text until we reach target vocab size."""
        print("Training BPE tokenizer...")
        word_freqs = self._get_word_freqs(text)

        # Initial vocab: every unique character in the data
        all_chars = set()
        for word in word_freqs:
            for ch in word:
                all_chars.add(ch)

        # We also need special tokens
        special_tokens = ["<pad>", "<unk>", "<eos>"]
        initial_size = len(all_chars) + len(special_tokens)
        num_merges = target_vocab_size - initial_size

        print(f"  Initial characters: {len(all_chars)}")
        print(f"  Merges to learn: {num_merges}")

        self.merges = []
        for i in range(num_merges):
            pairs = self._get_pair_counts(word_freqs)
            if not pairs:
                break

            best_pair = max(pairs, key=pairs.get)
            count = pairs[best_pair]

            if count < 2:  # no pair appears more than once, stop
                break

            word_freqs = self._merge_pair(word_freqs, best_pair)
            self.merges.append(best_pair)

            if (i + 1) % 500 == 0:
                print(f"  Merge {i+1}: {best_pair[0]} + {best_pair[1]} → {best_pair[0]+best_pair[1]} (count: {count})")

        # Build the final vocabulary
        # Start with special tokens at known indices
        self.vocab = {}
        for i, tok in enumerate(special_tokens):
            self.vocab[tok] = i

        # Add all characters
        for ch in sorted(all_chars):
            if ch not in self.vocab:
                self.vocab[ch] = len(self.vocab)

        # Add all merged tokens
        for pair in self.merges:
            merged = pair[0] + pair[1]
            if merged not in self.vocab:
                self.vocab[merged] = len(self.vocab)

        self.id_to_token = {v: k for k, v in self.vocab.items()}

        print(f"  Final vocab size: {len(self.vocab)}")
        print(f"  Learned {len(self.merges)} merges")
        return self

    def _tokenize_word(self, word):
        """Apply learned merges to a single word."""
        tokens = list(word) + ["</w>"]

        # Apply each merge rule in order
        for pair in self.merges:
            merged = pair[0] + pair[1]
            i = 0
            new_tokens = []
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        return tokens

    def encode(self, text):
        """Text → list of token IDs."""
        ids = []
        words = text.split()
        unk_id = self.vocab["<unk>"]
        cache = {}  # word → list of token IDs
        
        for word in words:
            if word == "<eos>":
                ids.append(self.vocab["<eos>"])
                continue
            
            if word not in cache:
                tokens = self._tokenize_word(word)
                cache[word] = [self.vocab.get(tok, unk_id) for tok in tokens]
            
            ids.extend(cache[word])
        
        return ids

    def decode(self, ids):
        """List of token IDs → text."""
        tokens = [self.id_to_token.get(i, "<unk>") for i in ids]
        # Join tokens and remove end-of-word markers, restoring spaces
        text = ""
        for tok in tokens:
            if tok in ("<pad>", "<unk>", "<eos>"):
                text += " " + tok
            elif tok.endswith("</w>"):
                text += tok[:-4] + " "  # remove </w>, add space
            else:
                text += tok
        return text.strip()


# ============================================
# Train the tokenizer on your data
# ============================================

text = " ".join(sample.lower() + " <eos>" for sample in dataset['train']['text'])

tokenizer = BPETokenizer()
tokenizer.train(text, target_vocab_size=8000)
vocab_size = len(tokenizer.vocab)

# See what it learned
print("\nExamples:")
for test in ["the cat sat on the mat", "unfortunately", "bioluminescence"]:
    tokens = tokenizer._tokenize_word(test) if " " not in test else sum([tokenizer._tokenize_word(w) for w in test.split()], [])
    print(f"  '{test}' → {tokens}")

# ============================================
# Encode all data
# ============================================

print("\nEncoding dataset...")
data = np.array(tokenizer.encode(text), dtype=np.int64)
print(f"Total tokens: {len(data):,}")

# ============================================
# Create training pairs
# ============================================

X, y = [], []
for i in range(0, len(data) - block_size, block_size):
    X.append(data[i:i + block_size])
    y.append(data[i + 1:i + block_size + 1])

X = np.array(X)
y = np.array(y)

split_idx = int(len(X) * 0.9)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

num_samples = X_train.shape[0]
steps_per_epoch = num_samples // batch_size

print(f"Train samples: {num_samples:,}")
print(f"Test samples: {len(X_test):,}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Vocab size: {vocab_size}")
print(f"Batch size: {batch_size}")

Training BPE tokenizer...
  Initial characters: 1423
  Merges to learn: 6574
  Merge 500: ow + n → own (count: 9396)
  Merge 1000: l + y.</w> → ly.</w> (count: 3685)
  Merge 1500: o + t → ot (count: 2181)
  Merge 2000: h + it</w> → hit</w> (count: 1492)
  Merge 2500: hon + e → hone (count: 1115)
  Merge 3000: be + ,</w> → be,</w> (count: 867)
  Merge 3500: europe + an</w> → european</w> (count: 709)
  Merge 4000: self + - → self- (count: 592)
  Merge 4500: m + er</w> → mer</w> (count: 501)
  Merge 5000: ie + ty</w> → iety</w> (count: 430)
  Merge 5500: - + l → -l (count: 379)
  Merge 6000: teach + ing</w> → teaching</w> (count: 334)
  Merge 6500: remember + ed</w> → remembered</w> (count: 297)
  Final vocab size: 8000
  Learned 6574 merges

Examples:
  'the cat sat on the mat' → ['the</w>', 'cat</w>', 'sat</w>', 'on</w>', 'the</w>', 'm', 'at</w>']
  'unfortunately' → ['unfortun', 'ately</w>']
  'bioluminescence' → ['bi', 'ol', 'um', 'in', 'es', 'c', 'ence</w>']

Encoding dataset...
Tot